# SmartSales — Info opzoeken

Gebruik deze cellen om snel SmartSales data op te zoeken voor ground truth verificatie.

**Secties:**
1. Auth
2. Locations — lijst / zoek op naam
3. Catalog items — lijst / zoek op code of titel
4. Orders — lijst / detail per ref of uid

## 1. Auth

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath('../..'))

import httpx
from dotenv import load_dotenv
load_dotenv()

from smartsales.auth import authenticate_from_env
ss_creds = authenticate_from_env()
SS_TOKEN = ss_creds.access_token
SS_BASE  = 'https://proxy-smartsales.easi.net/proxy/rest'

def ss(method: str, path: str, **kwargs):
    r = httpx.request(method, f'{SS_BASE}{path}',
                      headers={'Authorization': f'Bearer {SS_TOKEN}'}, **kwargs)
    r.raise_for_status()
    return r.json()

print('✓ SmartSales auth OK')

---
## 2. Locations

In [ ]:
# Alle locaties (gesorteerd op naam)
data = ss('GET', '/api/v3/location/list', params={'p': 'simple', 's': '200'})
locs = sorted(data.get('entries', []), key=lambda l: l.get('name', ''))
print(f'{len(locs)} locaties\n')
for l in locs:
    print(f"  {l.get('name','?'):<45} {l.get('city','?'):<15} code={l.get('code','?')}  uid={l.get('uid','')}")

In [ ]:
# Zoek locatie op naam (contains)
SEARCH = 'supplier'  # ← aanpassen

q = json.dumps({'name': f'contains:{SEARCH}'})
data = ss('GET', '/api/v3/location/list', params={'q': q, 'p': 'simple'})
locs = data.get('entries', [])
print(f'{len(locs)} resultaten voor "{SEARCH}"\n')
for l in locs:
    print(f"  {l.get('name','?'):<45} {l.get('city','?'):<15} code={l.get('code','?')}  uid={l.get('uid','')}")

In [ ]:
# Locatie detail op uid
UID = 'cac5b20d-285c-11f1-a2fe-005056010707'  # ← aanpassen

loc = ss('GET', f'/api/v3/location/{UID}')
for k, v in loc.items():
    if v not in (None, '', [], {}):
        print(f"  {k:<25} {v}")

---
## 3. Catalog items

In [ ]:
# Alle catalog items
data = ss('GET', '/api/v3/catalog/list', params={'p': 'simple', 's': '200'})
items = sorted(data.get('entries', []), key=lambda i: i.get('code', ''))
print(f'{len(items)} catalog items\n')
for i in items:
    print(f"  {i.get('code','?'):<20} {i.get('title','?'):<50} EUR {i.get('price',0):.2f}  uid={i.get('uid','')}")

In [ ]:
# Zoek catalog item op code of titel (client-side filter)
SEARCH = 'PROD-GTS'  # ← aanpassen (code of gedeelte van titel)

data = ss('GET', '/api/v3/catalog/list', params={'p': 'simple', 's': '200'})
matches = [
    i for i in data.get('entries', [])
    if SEARCH.lower() in (i.get('code') or '').lower()
    or SEARCH.lower() in (i.get('title') or '').lower()
]
print(f'{len(matches)} resultaten voor "{SEARCH}"\n')
for i in matches:
    print(f"  code={i.get('code','?')}  title={i.get('title','?')}  price={i.get('price',0)}  uid={i.get('uid','')}")
    if i.get('description'):
        print(f"    description: {i['description']}")

---
## 4. Orders

In [ ]:
# Alle orders (lijst, max 100)
data = ss('GET', '/api/v3/order/list', params={'p': 'simple', 's': '100'})
orders = data.get('entries', [])
print(f'{len(orders)} orders\n')
for o in orders:
    ref = o.get('internalReference') or o.get('externalReference') or '?'
    print(f"  {ref:<28} uid={o.get('uid','')}")

In [ ]:
# Order detail op internalReference
REF = 'ORD-SUP1-2026-001'  # ← aanpassen

q = json.dumps({'internalReference': f'eq:{REF}'})
data = ss('GET', '/api/v3/order/list', params={'q': q, 'p': 'simple'})
entries = data.get('entries', [])
if not entries:
    print(f'Niet gevonden: {REF}')
else:
    uid = entries[0]['uid']
    order = ss('GET', f'/api/v3/order/{uid}')
    sup  = (order.get('supplier') or {}).get('name', '?')
    cust = (order.get('customer') or {}).get('name', '?')
    print(f"ref      : {order.get('internalReference')}")
    print(f"uid      : {order.get('uid')}")
    print(f"supplier : {sup}")
    print(f"customer : {cust}")
    print(f"total    : EUR {order.get('total', 0):.2f}")
    print(f"status   : {order.get('approbationStatus')}")
    print(f"comments : {order.get('comments')}")
    print(f"\nItems:")
    for it in (order.get('items') or []):
        print(f"  {it.get('code','?'):<20} {it.get('description','?'):<45} x{it.get('quantity',0)}  @ EUR {it.get('price',0):.2f}  = EUR {it.get('totalPrice',0):.2f}")

In [ ]:
# Orders per supplier-locatie (zoek op naam)
SUPPLIER_NAME = 'supplier1'  # ← aanpassen

data = ss('GET', '/api/v3/order/list', params={'p': 'simple', 's': '100'})
all_orders = data.get('entries', [])

results = []
for entry in all_orders:
    order = ss('GET', f'/api/v3/order/{entry["uid"]}')
    sup_name = (order.get('supplier') or {}).get('name', '')
    if SUPPLIER_NAME.lower() in sup_name.lower():
        results.append(order)

print(f'{len(results)} orders voor supplier "{SUPPLIER_NAME}"\n')
total = 0
for o in results:
    ref   = o.get('internalReference') or '?'
    cust  = (o.get('customer') or {}).get('name', '?')
    amount = o.get('total') or 0
    total += amount
    print(f"  {ref:<28} customer={cust:<30} EUR {amount:>10,.2f}")
print(f"\n  Totaal: EUR {total:,.2f}")